# Live Model Testing — Real Current Market Data

This notebook loads your trained LSTM model and tests it against **real, current stock prices**.

**What this does:**
- Uploads your saved model + scalers from local machine
- Fetches the **latest real market data** (today's data via yfinance)
- Rebuilds identical features as used during training
- Predicts tomorrow's price using today's real data
- Backtests on the last 30 trading days to measure live accuracy
- Produces a full evaluation dashboard

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q yfinance pandas-ta scikit-learn tensorflow joblib plotly
print('All packages ready!')

## Cell 2 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json
import datetime
warnings.filterwarnings('ignore')

import yfinance as yf
import joblib
import pandas_ta as ta

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, r2_score
)
import tensorflow as tf
from tensorflow.keras.models import load_model

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

TODAY = datetime.date.today()
print(f'TensorFlow : {tf.__version__}')
print(f'Test Date  : {TODAY}')
print(f'GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}')

## Cell 3 — Upload Model Files from Your PC

Upload these 3 files saved from training:
- `stock_lstm_model.keras`
- `feature_scaler.pkl`
- `close_scaler.pkl`

In [ ]:
from google.colab import files

print('Upload your 3 saved files now:')
print('  1. stock_lstm_model.keras')
print('  2. feature_scaler.pkl')
print('  3. close_scaler.pkl')
print()

uploaded = files.upload()
print(f'\nUploaded files: {list(uploaded.keys())}')

## Cell 4 — Load Model + Scalers

In [ ]:
# Load model
model = load_model('stock_lstm_model.keras')
print('Model loaded')
model.summary()

# Load scalers
scaler       = joblib.load('feature_scaler.pkl')
close_scaler = joblib.load('close_scaler.pkl')
print('Scalers loaded')

# Model config — must match training
TARGET     = 'AAPL'
TICKERS    = ['AAPL', 'GOOG', 'MSFT', 'AMZN']
SEQ_LEN    = 60

FEATURE_COLS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_sig', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width', 'ATR_14',
    'OBV', 'VWAP',
    'HL_ratio', 'OC_ratio', 'Log_Return',
    'Volatility_5', 'Volatility_20',
    'Close_lag1', 'Close_lag2', 'Close_lag3', 'Close_lag4', 'Close_lag5'
]

close_idx = FEATURE_COLS.index('Close')
print(f'Target      : {TARGET}')
print(f'Seq length  : {SEQ_LEN}')
print(f'Features    : {len(FEATURE_COLS)}')

## Cell 5 — Fetch Real Current Market Data

In [ ]:
# Fetch enough history so we can build all technical indicators
# We need at least SEQ_LEN + 50 (for SMA_50) + buffer = ~120 days
LOOKBACK_DAYS = 180
start_date = (TODAY - datetime.timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')

print(f'Fetching real market data from {start_date} to {TODAY} ...')

raw = yf.download(TICKERS, start=start_date, auto_adjust=True, progress=True)
raw.columns = ['_'.join(col).strip() for col in raw.columns]
raw.index   = pd.to_datetime(raw.index)

# Extract target stock
ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df = pd.DataFrame({col: raw[f'{TARGET}_{col}'] for col in ohlcv_cols})
df = df.ffill().bfill().sort_index()

print(f'\nReal data shape   : {df.shape}')
print(f'Date range        : {df.index.min().date()} to {df.index.max().date()}')
print(f'Latest close      : ${df["Close"].iloc[-1]:.2f}')
print(f'Previous close    : ${df["Close"].iloc[-2]:.2f}')
print(f'Day change        : {((df["Close"].iloc[-1] / df["Close"].iloc[-2]) - 1) * 100:+.2f}%')
print(f'\nLatest 5 rows:')
df.tail()

## Cell 6 — Build Features (Identical to Training)

In [ ]:
def build_features(df):
    feat = df.copy()

    # Trend
    feat['SMA_20']    = ta.sma(feat['Close'], length=20)
    feat['SMA_50']    = ta.sma(feat['Close'], length=50)
    feat['EMA_12']    = ta.ema(feat['Close'], length=12)
    feat['EMA_26']    = ta.ema(feat['Close'], length=26)

    # Momentum
    feat['RSI_14']    = ta.rsi(feat['Close'], length=14)
    macd              = ta.macd(feat['Close'], fast=12, slow=26, signal=9)
    feat['MACD']      = macd['MACD_12_26_9']
    feat['MACD_sig']  = macd['MACDs_12_26_9']
    feat['MACD_hist'] = macd['MACDh_12_26_9']

    # Volatility
    bb                = ta.bbands(feat['Close'], length=20, std=2)
    feat['BB_upper']  = bb['BBU_20_2.0']
    feat['BB_lower']  = bb['BBL_20_2.0']
    feat['BB_mid']    = bb['BBM_20_2.0']
    feat['BB_width']  = (feat['BB_upper'] - feat['BB_lower']) / feat['BB_mid']
    feat['ATR_14']    = ta.atr(feat['High'], feat['Low'], feat['Close'], length=14)

    # Volume
    feat['OBV']       = ta.obv(feat['Close'], feat['Volume'])
    feat['VWAP']      = (
        feat['Volume'] * (feat['High'] + feat['Low'] + feat['Close']) / 3
    ).cumsum() / feat['Volume'].cumsum()

    # Price ratios
    feat['HL_ratio']      = feat['High'] / feat['Low']
    feat['OC_ratio']      = feat['Open'] / feat['Close']
    feat['Log_Return']    = np.log(feat['Close'] / feat['Close'].shift(1))
    feat['Volatility_5']  = feat['Log_Return'].rolling(5).std()
    feat['Volatility_20'] = feat['Log_Return'].rolling(20).std()

    # Lags
    for lag in range(1, 6):
        feat[f'Close_lag{lag}'] = feat['Close'].shift(lag)

    feat.dropna(inplace=True)
    return feat

feat = build_features(df)
print(f'Feature set shape : {feat.shape}')
print(f'Last date         : {feat.index[-1].date()}')
print(f'Available for seq : {len(feat)} rows (need at least {SEQ_LEN})')

assert len(feat) >= SEQ_LEN, f'Not enough data! Got {len(feat)}, need {SEQ_LEN}'
print('Feature check passed!')

## Cell 7 — TODAY's Prediction (Next Trading Day)

In [ ]:
# Take the last SEQ_LEN rows of real live data
live_window_raw    = feat[FEATURE_COLS].values[-SEQ_LEN:]
live_window_scaled = scaler.transform(live_window_raw)
live_input         = live_window_scaled[np.newaxis]   # (1, 60, 29)

# Predict
pred_scaled  = model.predict(live_input, verbose=0)[0, 0]
pred_price   = close_scaler.inverse_transform([[pred_scaled]])[0, 0]

last_close   = feat['Close'].iloc[-1]
last_date    = feat.index[-1]
change_usd   = pred_price - last_close
change_pct   = change_usd / last_close * 100

# Next business day
next_day = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=1)[0]

# Support / Resistance from BB
support    = feat['BB_lower'].iloc[-1]
resistance = feat['BB_upper'].iloc[-1]
rsi_now    = feat['RSI_14'].iloc[-1]
macd_now   = feat['MACD'].iloc[-1]
macd_sig   = feat['MACD_sig'].iloc[-1]

# Signal
if change_pct > 1.5:
    signal, color = 'STRONG BUY', '#10b981'
elif change_pct > 0:
    signal, color = 'BUY', '#34d399'
elif change_pct > -1.5:
    signal, color = 'SELL', '#f87171'
else:
    signal, color = 'STRONG SELL', '#ef4444'

print('=' * 60)
print(f'  LIVE PREDICTION FOR {TARGET}')
print('=' * 60)
print(f'  As of          : {last_date.date()} (last trading day)')
print(f'  Last Close     : ${last_close:.2f}')
print(f'  Prediction for : {next_day.date()}')
print(f'  Predicted Close: ${pred_price:.2f}')
print(f'  Expected Move  : {change_usd:+.2f} USD  ({change_pct:+.2f}%)')
print(f'  Signal         : {signal}')
print('-' * 60)
print(f'  RSI (14)       : {rsi_now:.1f}  ({"Overbought" if rsi_now > 70 else "Oversold" if rsi_now < 30 else "Neutral"})')
print(f'  MACD           : {macd_now:.3f}  Signal: {macd_sig:.3f}  ({"Bullish" if macd_now > macd_sig else "Bearish"})')
print(f'  BB Support     : ${support:.2f}')
print(f'  BB Resistance  : ${resistance:.2f}')
print('=' * 60)

## Cell 8 — 30-Day Backtest: Predictions vs Actual Real Prices

In [ ]:
# Backtest over the last 30 trading days
# For each day D, use [D-60 : D] to predict D+1 and compare vs actual

BACKTEST_DAYS = 30

# We need at least SEQ_LEN + BACKTEST_DAYS rows
if len(feat) < SEQ_LEN + BACKTEST_DAYS:
    BACKTEST_DAYS = len(feat) - SEQ_LEN - 1
    print(f'Adjusted backtest window to {BACKTEST_DAYS} days (limited data)')

all_data      = feat[FEATURE_COLS].values
all_scaled    = scaler.transform(all_data)

bt_actuals    = []
bt_preds      = []
bt_dates      = []

# Offset from end: we predict days [-BACKTEST_DAYS:] using prior window
start_idx = len(all_scaled) - BACKTEST_DAYS - 1

for i in range(BACKTEST_DAYS):
    idx = start_idx + i
    window = all_scaled[idx - SEQ_LEN + 1 : idx + 1]   # SEQ_LEN rows ending at idx
    actual = feat['Close'].iloc[idx + 1]                # next day actual
    date   = feat.index[idx + 1]

    pred_s = model.predict(window[np.newaxis], verbose=0)[0, 0]
    pred_r = close_scaler.inverse_transform([[pred_s]])[0, 0]

    bt_actuals.append(actual)
    bt_preds.append(pred_r)
    bt_dates.append(date)

bt_actuals = np.array(bt_actuals)
bt_preds   = np.array(bt_preds)

# Metrics
rmse = np.sqrt(mean_squared_error(bt_actuals, bt_preds))
mae  = mean_absolute_error(bt_actuals, bt_preds)
mape = mean_absolute_percentage_error(bt_actuals, bt_preds) * 100
r2   = r2_score(bt_actuals, bt_preds)
da   = np.mean(np.sign(np.diff(bt_actuals)) == np.sign(np.diff(bt_preds))) * 100

# How many days within 1% / 2% / 5% of actual?
within_1pct = np.mean(np.abs((bt_preds - bt_actuals) / bt_actuals) < 0.01) * 100
within_2pct = np.mean(np.abs((bt_preds - bt_actuals) / bt_actuals) < 0.02) * 100
within_5pct = np.mean(np.abs((bt_preds - bt_actuals) / bt_actuals) < 0.05) * 100

print('=' * 60)
print(f'  BACKTEST RESULTS — Last {BACKTEST_DAYS} Trading Days')
print('=' * 60)
print(f'  RMSE                  : ${rmse:.4f}')
print(f'  MAE                   : ${mae:.4f}')
print(f'  MAPE                  : {mape:.2f}%')
print(f'  R-Squared             : {r2:.4f}')
print(f'  Directional Accuracy  : {da:.1f}%')
print(f'  Within 1% of actual   : {within_1pct:.1f}% of days')
print(f'  Within 2% of actual   : {within_2pct:.1f}% of days')
print(f'  Within 5% of actual   : {within_5pct:.1f}% of days')
print('=' * 60)

# Grade the model
if mape < 2 and da > 60:
    grade = 'EXCELLENT'
elif mape < 4 and da > 55:
    grade = 'GOOD'
elif mape < 7:
    grade = 'FAIR'
else:
    grade = 'NEEDS RETRAINING'
print(f'  Model Grade           : {grade}')

## Cell 9 — Visual Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0f172a')

# ── 1. Backtest: Actual vs Predicted ──────────────────────────────────────
ax1 = fig.add_subplot(3, 2, (1, 2))
ax1.set_facecolor('#1e293b')
ax1.plot(bt_dates, bt_actuals, color='#00d4ff', lw=2.5, label='Actual Price', zorder=3)
ax1.plot(bt_dates, bt_preds,   color='#f43f5e', lw=2, ls='--',
         label='LSTM Prediction', zorder=3)
ax1.fill_between(bt_dates, bt_actuals, bt_preds, alpha=0.15, color='#f59e0b')

# Mark each prediction point
for i, (d, a, p) in enumerate(zip(bt_dates, bt_actuals, bt_preds)):
    color = '#10b981' if abs(p - a) / a < 0.02 else '#f59e0b' if abs(p - a) / a < 0.05 else '#ef4444'
    ax1.scatter(d, p, color=color, s=30, zorder=5)

ax1.set_title(f'{TARGET} — {BACKTEST_DAYS}-Day Live Backtest\n'
              f'MAPE={mape:.2f}%   R²={r2:.4f}   Dir.Acc={da:.1f}%',
              fontsize=13, fontweight='bold', color='white')
ax1.set_ylabel('Price (USD)', color='white')
ax1.tick_params(colors='white')
ax1.legend(fontsize=10, facecolor='#1e293b', labelcolor='white')
for spine in ax1.spines.values():
    spine.set_edgecolor('#334155')

# ── 2. Daily Prediction Error ──────────────────────────────────────────────
ax2 = fig.add_subplot(3, 2, 3)
ax2.set_facecolor('#1e293b')
errors_pct = (bt_preds - bt_actuals) / bt_actuals * 100
bar_colors = ['#10b981' if abs(e) < 1 else '#f59e0b' if abs(e) < 3 else '#ef4444'
              for e in errors_pct]
ax2.bar(range(len(errors_pct)), errors_pct, color=bar_colors, alpha=0.85)
ax2.axhline(0,  color='white', lw=0.8, ls='--')
ax2.axhline(2,  color='#f59e0b', lw=0.8, ls=':', alpha=0.7)
ax2.axhline(-2, color='#f59e0b', lw=0.8, ls=':', alpha=0.7)
ax2.set_title('Daily Prediction Error (%)', fontsize=11, color='white', fontweight='bold')
ax2.set_xlabel('Trading Day', color='white')
ax2.set_ylabel('Error %', color='white')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_edgecolor('#334155')

g1 = mpatches.Patch(color='#10b981', label='<1% error')
g2 = mpatches.Patch(color='#f59e0b', label='1-3% error')
g3 = mpatches.Patch(color='#ef4444', label='>3% error')
ax2.legend(handles=[g1, g2, g3], fontsize=8, facecolor='#1e293b', labelcolor='white')

# ── 3. Accuracy within bands ───────────────────────────────────────────────
ax3 = fig.add_subplot(3, 2, 4)
ax3.set_facecolor('#1e293b')
bands  = ['Within 1%', 'Within 2%', 'Within 5%']
values = [within_1pct, within_2pct, within_5pct]
bcolors = ['#10b981', '#f59e0b', '#a855f7']
bars = ax3.barh(bands, values, color=bcolors, alpha=0.85)
for bar, val in zip(bars, values):
    ax3.text(min(val + 1, 95), bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}%', va='center', color='white', fontweight='bold')
ax3.axvline(50, color='white', lw=0.8, ls='--', alpha=0.5)
ax3.set_xlim(0, 105)
ax3.set_title('Prediction Accuracy Bands', fontsize=11, color='white', fontweight='bold')
ax3.set_xlabel('% of Days', color='white')
ax3.tick_params(colors='white')
for spine in ax3.spines.values():
    spine.set_edgecolor('#334155')

# ── 4. RSI Chart ───────────────────────────────────────────────────────────
ax4 = fig.add_subplot(3, 2, 5)
ax4.set_facecolor('#1e293b')
rsi_series = feat['RSI_14'].iloc[-BACKTEST_DAYS:]
ax4.plot(rsi_series.index, rsi_series.values, color='#a855f7', lw=2)
ax4.axhline(70, color='#ef4444', ls='--', lw=1, alpha=0.8, label='Overbought (70)')
ax4.axhline(30, color='#10b981', ls='--', lw=1, alpha=0.8, label='Oversold (30)')
ax4.axhline(50, color='white',   ls=':',  lw=0.8, alpha=0.5)
ax4.fill_between(rsi_series.index, rsi_series.values, 70,
                 where=(rsi_series.values >= 70), alpha=0.15, color='#ef4444')
ax4.fill_between(rsi_series.index, rsi_series.values, 30,
                 where=(rsi_series.values <= 30), alpha=0.15, color='#10b981')
ax4.set_ylim(0, 100)
ax4.set_title('RSI-14 (Current Period)', fontsize=11, color='white', fontweight='bold')
ax4.set_ylabel('RSI', color='white')
ax4.tick_params(colors='white')
ax4.legend(fontsize=8, facecolor='#1e293b', labelcolor='white')
for spine in ax4.spines.values():
    spine.set_edgecolor('#334155')

# ── 5. Metrics scorecard ───────────────────────────────────────────────────
ax5 = fig.add_subplot(3, 2, 6)
ax5.set_facecolor('#1e293b')
ax5.axis('off')

score_data = [
    ['METRIC',                'VALUE',          'RATING'],
    ['RMSE',                  f'${rmse:.2f}',    'Good' if rmse < 5 else 'Fair'],
    ['MAE',                   f'${mae:.2f}',     'Good' if mae < 3 else 'Fair'],
    ['MAPE',                  f'{mape:.2f}%',    'Good' if mape < 4 else 'Fair'],
    ['R-Squared',             f'{r2:.4f}',       'Good' if r2 > 0.95 else 'Fair'],
    ['Directional Accuracy',  f'{da:.1f}%',      'Good' if da > 55 else 'Fair'],
    ['Within 1% Band',        f'{within_1pct:.0f}%', ''],
    ['Within 2% Band',        f'{within_2pct:.0f}%', ''],
    ['Model Grade',           grade,             ''],
]

col_widths = [0.5, 0.3, 0.2]
row_colors = ['#334155'] + ['#1e293b' if i % 2 == 0 else '#263044' for i in range(8)]
for row_i, row in enumerate(score_data):
    for col_i, (text, w) in enumerate(zip(row, [0.0, 0.5, 0.8])):
        fw = 'bold' if row_i == 0 else 'normal'
        fc = '#00d4ff' if row_i == 0 else \
             '#10b981' if text == 'Good' else \
             '#f59e0b' if text == 'Fair' else 'white'
        ax5.text(w, 1 - row_i * 0.115,
                 text, transform=ax5.transAxes,
                 fontsize=9.5, color=fc, fontweight=fw, va='top')

ax5.set_title('Model Scorecard', fontsize=11, color='white', fontweight='bold')

plt.suptitle(f'{TARGET} LSTM — Live Market Test Dashboard   |   {TODAY}',
             fontsize=16, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('live_test_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()
print('Dashboard saved -> live_test_dashboard.png')

## Cell 10 — Interactive Plotly Chart

In [ ]:
# Full OHLC candlestick + prediction overlay (interactive)
hist = feat.iloc[-BACKTEST_DAYS - 5:].copy()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.2, 0.2],
    subplot_titles=[f'{TARGET} — Candlestick + LSTM Predictions', 'RSI-14', 'MACD']
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=hist.index,
    open=hist['Open'],  high=hist['High'],
    low=hist['Low'],    close=hist['Close'],
    name='OHLC', increasing_line_color='#10b981', decreasing_line_color='#ef4444'
), row=1, col=1)

# Predictions overlay
bt_df = pd.Series(bt_preds, index=bt_dates)
fig.add_trace(go.Scatter(
    x=bt_df.index, y=bt_df.values,
    mode='lines+markers',
    name='LSTM Prediction',
    line=dict(color='#f43f5e', width=2, dash='dot'),
    marker=dict(size=6)
), row=1, col=1)

# Tomorrow prediction point
fig.add_trace(go.Scatter(
    x=[next_day], y=[pred_price],
    mode='markers+text',
    name=f'Tomorrow Pred ${pred_price:.2f}',
    marker=dict(size=14, color='#f59e0b', symbol='star'),
    text=[f'${pred_price:.2f}'],
    textposition='top center'
), row=1, col=1)

# Bollinger Bands
fig.add_trace(go.Scatter(
    x=hist.index, y=hist['BB_upper'],
    name='BB Upper', line=dict(color='rgba(168,85,247,0.4)', width=1, dash='dash')
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=hist.index, y=hist['BB_lower'],
    name='BB Lower', fill='tonexty',
    fillcolor='rgba(168,85,247,0.05)',
    line=dict(color='rgba(168,85,247,0.4)', width=1, dash='dash')
), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(
    x=hist.index, y=hist['RSI_14'],
    name='RSI', line=dict(color='#a855f7', width=1.5)
), row=2, col=1)
fig.add_hline(y=70, line_color='#ef4444', line_dash='dash', opacity=0.6, row=2, col=1)
fig.add_hline(y=30, line_color='#10b981', line_dash='dash', opacity=0.6, row=2, col=1)

# MACD
fig.add_trace(go.Scatter(
    x=hist.index, y=hist['MACD'],
    name='MACD', line=dict(color='#00d4ff', width=1.5)
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=hist.index, y=hist['MACD_sig'],
    name='Signal', line=dict(color='#f59e0b', width=1.5)
), row=3, col=1)
hist['MACD_bar_color'] = np.where(hist['MACD_hist'] >= 0, '#10b981', '#ef4444')
fig.add_trace(go.Bar(
    x=hist.index, y=hist['MACD_hist'],
    name='MACD Hist', marker_color=hist['MACD_bar_color'], opacity=0.7
), row=3, col=1)

fig.update_layout(
    title=f'{TARGET} Live Test Dashboard — {TODAY}',
    template='plotly_dark',
    height=900,
    xaxis_rangeslider_visible=False,
    legend=dict(orientation='h', y=-0.05)
)
fig.show()

## Cell 11 — Full Scenario Test (Bull / Bear / Sideways Simulation)

In [ ]:
# Stress test: modify current window to simulate different market conditions
# and see how the model responds

scenarios = {
    'Current Market':  1.00,
    'Bull (+10%)':     1.10,
    'Strong Bull (+20%)': 1.20,
    'Bear (-10%)':     0.90,
    'Crash (-20%)':    0.80,
    'Sideways (no change)': 1.00,
}

print('=' * 65)
print(f'  SCENARIO STRESS TEST — {TARGET}')
print('=' * 65)
print(f'{"Scenario":<30} {"Simulated Close":>16} {"Predicted Next":>16} {"Change":>10}')
print('-' * 65)

results = []
base_window = feat[FEATURE_COLS].values[-SEQ_LEN:].copy()

for scenario_name, multiplier in scenarios.items():
    # Adjust the Close column in the last row
    sim_window = base_window.copy()
    if scenario_name != 'Current Market':
        # Scale the price columns in the last 5 rows
        price_indices = [FEATURE_COLS.index(c) for c in ['Open','High','Low','Close']]
        sim_window[-5:, price_indices] *= multiplier

    sim_scaled = scaler.transform(sim_window)
    pred_s = model.predict(sim_scaled[np.newaxis], verbose=0)[0, 0]
    pred_r = close_scaler.inverse_transform([[pred_s]])[0, 0]

    sim_close = last_close * multiplier
    chg = (pred_r - sim_close) / sim_close * 100
    results.append((scenario_name, sim_close, pred_r, chg))

    dir_sym = 'UP  ' if chg > 0 else 'DOWN'
    print(f'{scenario_name:<30} ${sim_close:>14.2f}  ${pred_r:>14.2f}  {chg:>+8.2f}% {dir_sym}')

print('=' * 65)

# Bar chart of scenario results
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

names     = [r[0] for r in results]
preds     = [r[2] for r in results]
sim_close = [r[1] for r in results]
bar_colors = ['#10b981' if r[3] > 0 else '#ef4444' for r in results]

x = np.arange(len(names))
ax.bar(x - 0.2, sim_close, 0.35, label='Simulated Close', color='#00d4ff', alpha=0.7)
ax.bar(x + 0.2, preds,     0.35, label='LSTM Prediction',  color=bar_colors, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', color='white', fontsize=9)
ax.set_ylabel('Price (USD)', color='white')
ax.set_title(f'{TARGET} — Model Response to Market Scenarios', fontsize=13,
             color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#1e293b', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

plt.tight_layout()
plt.savefig('scenario_test.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## Cell 12 — Compare Predictions Across All 4 Stocks

In [ ]:
# Run the same model and scalers on all 4 stocks
# (Note: model was trained on AAPL; results on others are illustrative)

multi_results = {}

for ticker in TICKERS:
    try:
        ticker_df = pd.DataFrame({
            col: raw[f'{ticker}_{col}'] for col in ohlcv_cols
        }).ffill().bfill().sort_index()

        ticker_feat = build_features(ticker_df)

        if len(ticker_feat) < SEQ_LEN:
            print(f'{ticker}: insufficient data, skipping')
            continue

        window_raw    = ticker_feat[FEATURE_COLS].values[-SEQ_LEN:]
        window_scaled = scaler.transform(window_raw)
        pred_s        = model.predict(window_scaled[np.newaxis], verbose=0)[0, 0]
        pred_r        = close_scaler.inverse_transform([[pred_s]])[0, 0]
        last_c        = ticker_feat['Close'].iloc[-1]
        chg           = (pred_r - last_c) / last_c * 100

        multi_results[ticker] = {
            'last_close': last_c,
            'prediction': pred_r,
            'change_pct': chg,
            'signal': 'BUY' if chg > 0 else 'SELL'
        }
    except Exception as e:
        print(f'{ticker}: error - {e}')

print('=' * 70)
print(f'  MULTI-STOCK PREDICTION SUMMARY — Next Trading Day')
print('=' * 70)
print(f'{"Ticker":<8} {"Last Close":>12} {"Prediction":>12} {"Change %":>10} {"Signal":>10}')
print('-' * 70)
for ticker, r in multi_results.items():
    sig_symbol = 'BUY' if r['signal'] == 'BUY' else 'SELL'
    print(f'{ticker:<8} ${r["last_close"]:>10.2f}  ${r["prediction"]:>10.2f}'
          f'  {r["change_pct"]:>+9.2f}%  {sig_symbol:>10}')
print('=' * 70)

## Cell 13 — Save Live Test Report

In [ ]:
import json

report = {
    'test_date':         str(TODAY),
    'target_stock':      TARGET,
    'last_close':        round(float(last_close), 2),
    'predicted_price':   round(float(pred_price), 2),
    'predicted_change':  round(float(change_pct), 2),
    'signal':            signal,
    'rsi':               round(float(rsi_now), 2),
    'macd':              round(float(macd_now), 4),
    'bb_support':        round(float(support), 2),
    'bb_resistance':     round(float(resistance), 2),
    'backtest_metrics': {
        'days':          BACKTEST_DAYS,
        'rmse':          round(float(rmse), 4),
        'mae':           round(float(mae), 4),
        'mape':          round(float(mape), 4),
        'r2':            round(float(r2), 4),
        'directional_accuracy': round(float(da), 2),
        'within_1pct':   round(float(within_1pct), 1),
        'within_2pct':   round(float(within_2pct), 1),
        'within_5pct':   round(float(within_5pct), 1),
        'grade':         grade
    },
    'multi_stock': {
        ticker: {
            'last_close': round(float(r['last_close']), 2),
            'prediction': round(float(r['prediction']), 2),
            'change_pct': round(float(r['change_pct']), 2),
            'signal':     r['signal']
        } for ticker, r in multi_results.items()
    }
}

with open('live_test_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print('\nReport saved -> live_test_report.json')

# Download results
try:
    from google.colab import files
    for fname in ['live_test_dashboard.png', 'scenario_test.png', 'live_test_report.json']:
        files.download(fname)
except ImportError:
    pass